# Setup Iniziale

In [1]:
# SEZIONE 0 — Setup e parametri
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- Parametri globali (tutto in un punto solo) ---
REPO_URL = "https://github.com/d2ski/football-transfers-data.git"
REPO_DIR = "football-transfers-data"

# Tetto per rimuovere SOLO i valori-spazzatura (placeholder 500M+),
# non gli outlier veri (record 2009-2021 = Neymar 222M).
FEE_CAP = 250_000_000

# Le 7 leghe realmente coperte dal dataset (mercato completo).
CORE_LEAGUES = ["IT1", "GB1", "PO1", "ES1", "FR1", "NL1", "L1"]

# Cartella output per i CSV destinati a R
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

In [2]:
# Cloniamo il repository dei dati (una tantum)
if not Path(REPO_DIR).exists():
    os.system(f"git clone {REPO_URL}")
else:
    print("Repo già presente.")

In [3]:
# unico CSV rilevante è transfers.csv
csv_files = list(Path(REPO_DIR).rglob("*.csv"))
transfers_raw = pd.read_csv([f for f in csv_files if f.stem == "transfers"][0])

print("transfers.csv:", transfers_raw.shape)
print("Colonne:", transfers_raw.columns.tolist())

transfers.csv: (70006, 23)
Colonne: ['league', 'season', 'window', 'team_id', 'team_name', 'team_country', 'dir', 'player_id', 'player_name', 'player_age', 'player_nation', 'player_nation2', 'player_pos', 'counter_team_id', 'counter_team_name', 'counter_team_country', 'transfer_fee_amnt', 'market_val_amnt', 'is_free', 'is_loan', 'is_loan_end', 'is_retired', 'transfer_id']


# Pulizia Base

In [4]:
# SEZIONE 1 — Pulizia base
base = transfers_raw.copy()

# 1) Booleani robusti (nel CSV possono arrivare come stringhe)
for col in ["is_loan", "is_loan_end", "is_free", "is_retired"]:
    if base[col].dtype == object:
        base[col] = base[col].astype(str).str.lower().isin(["true", "1", "yes", "t"])
    else:
        base[col] = base[col].astype(bool)

# 2) Tipi numerici decisi QUI una volta sola (ID inclusi -> Int64).
#    Nota: counter_team_id contiene "Retired" per i ritiri -> diventa <NA> (corretto:
#    i ritiri non hanno fee, non entrano in Y né negli archi).
base["season"]            = pd.to_numeric(base["season"], errors="coerce").astype("Int64")
base["transfer_fee_amnt"] = pd.to_numeric(base["transfer_fee_amnt"], errors="coerce")
base["market_val_amnt"]   = pd.to_numeric(base["market_val_amnt"], errors="coerce")
for col in ["team_id", "counter_team_id", "player_id"]:
    base[col] = pd.to_numeric(base[col], errors="coerce").astype("Int64")

# 3) Rimuoviamo la spazzatura economica (tiene NaN e fee <= cap)
n_before = len(base)
base = base[~(base["transfer_fee_amnt"] > FEE_CAP)].copy()
print(f"Righe-spazzatura rimosse (fee > {FEE_CAP:,}): {n_before - len(base)}")

# 4) De-dup per transfer_id
base = base.drop_duplicates(subset="transfer_id").copy()
print("base:", base.shape)

Righe-spazzatura rimosse (fee > 250,000,000): 35
base: (58607, 23)


# Ricostruzione Semantica

In [5]:
# SEZIONE 2 — Ricostruzione semantica -> `owns`
# dir == "left": team CEDE  -> team=venditore, counter=acquirente
# dir == "in":   team RICEVE -> team=acquirente, counter=venditore
seller = np.where(base["dir"] == "left", base["team_id"], base["counter_team_id"])
buyer  = np.where(base["dir"] == "left", base["counter_team_id"], base["team_id"])
base["seller_id"] = pd.array(seller, dtype="Int64")
base["buyer_id"]  = pd.array(buyer,  dtype="Int64")

# Escludi i prestiti -> solo passaggi di proprietà reali
owns = base[(~base["is_loan"]) & (~base["is_loan_end"])].copy()

print("owns (proprietà, no prestiti):", owns.shape)
print("Prestiti esclusi:", len(base) - len(owns))

owns (proprietà, no prestiti): (30774, 25)
Prestiti esclusi: 27833


# Perimetro Core

In [6]:
# SEZIONE 3 — Perimetro core -> `core_keys`, `core_set`
# Un club è core in una stagione se compare come `team` in una lega coperta.
core_keys = (base.loc[base["league"].isin(CORE_LEAGUES), ["team_id", "season"]]
             .dropna()
             .drop_duplicates()
             .rename(columns={"team_id": "club_id"})
             .reset_index(drop=True))

# Insieme di tuple (int, int): confronto veloce e senza mismatch di tipo
core_set = set(zip(core_keys["club_id"].astype(int), core_keys["season"].astype(int)))

print("Coppie (club, stagione) core:", len(core_keys))
print("Club core distinti:", core_keys["club_id"].nunique())

Coppie (club, stagione) core: 1728
Club core distinti: 239


# Vista Nodi: esito Y

In [7]:
# ============================================================
# SEZIONE 4 — Vista nodi
# 4a) Esito Y = somma margini (prezzo - valore di mercato) sulle vendite
# ============================================================
sales = owns[(owns["transfer_fee_amnt"].notna()) & (owns["transfer_fee_amnt"] > 0)].copy()
sales = sales.rename(columns={"seller_id": "club_id"})

# margine definito solo dove il market value è valido; le altre vendite escluse da Y
sales["has_mv"] = sales["market_val_amnt"] > 0
sales["margin"] = np.where(sales["has_mv"],
                           sales["transfer_fee_amnt"] - sales["market_val_amnt"],
                           np.nan)

Y = (sales[sales["has_mv"]]
     .groupby(["club_id", "season"])
     .agg(Y_margin_sum=("margin", "sum"),
          n_sales_Y=("transfer_id", "count"))
     .reset_index())

print("Vendite onerose totali:", len(sales))
print("Di cui con market value valido:", int(sales["has_mv"].sum()),
      f"({100*sales['has_mv'].mean():.1f}%)")
print("Righe Y (club-stagione con almeno una vendita valida):", len(Y))

Vendite onerose totali: 9171
Di cui con market value valido: 7649 (83.4%)
Righe Y (club-stagione con almeno una vendita valida): 3678


# Attributo baseline: spesa in acquisti per club-stagione

In [8]:
# ============================================================
# 4b) Attributo baseline: spesa in acquisti per club-stagione
# ============================================================
buys = owns[(owns["transfer_fee_amnt"].notna()) & (owns["transfer_fee_amnt"] > 0)].copy()

spend = (buys
         .groupby(["buyer_id", "season"])
         .agg(spend_sum=("transfer_fee_amnt", "sum"),
              n_buys=("transfer_id", "count"))
         .reset_index()
         .rename(columns={"buyer_id": "club_id"}))

print("Righe spend (club-stagione con almeno un acquisto):", len(spend))
print(spend.sort_values("spend_sum", ascending=False).head(5))

Righe spend (club-stagione con almeno un acquisto): 2870
      club_id  season    spend_sum  n_buys
509       131    2017  375100000.0       7
1135      418    2019  353500000.0       7
812       281    2017  317500000.0      10
511       131    2019  301000000.0       8
1373      631    2017  260500000.0       9


# Numero di vendite totali

In [9]:
# 4c) N. vendite totali + quota costo-zero (proxy vivaio/censura)
# Universo acquisti (anche gratuiti): per rintracciare se una vendita ha un acquisto anteriore
purchases = (owns[["buyer_id", "player_id", "season"]]
             .dropna()
             .rename(columns={"buyer_id": "club_id", "season": "buy_season"}))

# Per ogni vendita: esiste un acquisto dello STESSO club, PRIMA (o nella stessa stagione)?
trace = sales.merge(purchases, on=["club_id", "player_id"], how="left")
trace["valid_prior_buy"] = trace["buy_season"].notna() & (trace["buy_season"] <= trace["season"])
has_prior = trace.groupby("transfer_id")["valid_prior_buy"].any()

sales = sales.merge(has_prior.rename("has_prior_buy"), on="transfer_id", how="left")
sales["has_prior_buy"] = sales["has_prior_buy"].fillna(False)

# Aggreghiamo per club-stagione: n. vendite totali e quota costo-zero
sell_all = (sales
            .groupby(["club_id", "season"])
            .agg(n_sales_tot=("transfer_id", "count"),
                 n_costzero=("has_prior_buy", lambda s: int((~s).sum())))
            .reset_index())
sell_all["costzero_share"] = sell_all["n_costzero"] / sell_all["n_sales_tot"]

# Verifica: la quota costo-zero complessiva (deve essere ~67% sul valore, come già visto)
val_tot  = sales["transfer_fee_amnt"].sum()
val_zero = sales.loc[~sales["has_prior_buy"], "transfer_fee_amnt"].sum()
print("Righe sell_all:", len(sell_all))
print(f"Quota costo-zero sul VALORE: {100*val_zero/val_tot:.1f}%")
print(f"Quota costo-zero sul CONTEGGIO: {100*(~sales['has_prior_buy']).mean():.1f}%")

Righe sell_all: 4020
Quota costo-zero sul VALORE: 33.5%
Quota costo-zero sul CONTEGGIO: 48.6%


*Diagnostica esplorativa*

In [10]:
# controlliamo se sales si è gonfiato nel merge
print("Righe sales:", len(sales))
print("transfer_id unici in sales:", sales["transfer_id"].nunique())
print("→ sales ha duplicati di transfer_id?", len(sales) != sales["transfer_id"].nunique())

# verifichiamo se purchases ha righe duplicate (stesso club, player, stagione)
print("\nRighe purchases:", len(purchases))
print("purchases duplicati (club_id, player_id, buy_season):",
      purchases.duplicated().sum())

# Controprova sul merge trace
print("\nRighe trace (dopo merge):", len(trace))
print("→ il merge ha moltiplicato le righe?", len(trace), "vs sales", len(sales))

Righe sales: 9171
transfer_id unici in sales: 9171
→ sales ha duplicati di transfer_id? False

Righe purchases: 30037
purchases duplicati (club_id, player_id, buy_season): 14

Righe trace (dopo merge): 9426
→ il merge ha moltiplicato le righe? 9426 vs sales 9171


In [11]:
# gradiente per stagione

per_stagione = (sales
    .assign(cost_zero=~sales["has_prior_buy"])
    .groupby("season")
    .agg(n=("transfer_id", "size"),
         pct_costzero=("cost_zero", lambda s: round(100*s.mean(), 1)))
    .reset_index())
print(per_stagione.to_string(index=False))

 season   n  pct_costzero
   2009 615          96.7
   2010 619          85.1
   2011 753          70.3
   2012 686          53.2
   2013 699          44.5
   2014 675          40.7
   2015 787          39.0
   2016 783          37.3
   2017 838          34.7
   2018 834          38.0
   2019 853          34.8
   2020 553          36.9
   2021 476          31.7


# Creazione Tabella Nodo-Stagione

In [12]:
# 4d) Unione: tabella nodo-stagione completa (tutti i club)
node_season = (Y
    .merge(spend,    on=["club_id", "season"], how="outer")
    .merge(sell_all, on=["club_id", "season"], how="outer"))

# Assenza di attività -> 0 reale, non NaN (incluso costzero_share: 0 vendite -> 0)
fill0 = ["Y_margin_sum", "n_sales_Y", "spend_sum", "n_buys",
         "n_sales_tot", "n_costzero", "costzero_share"]
node_season[fill0] = node_season[fill0].fillna(0)

print("Tabella nodo-stagione (tutti i club):", node_season.shape)
print("Club distinti:", node_season["club_id"].nunique())

# Sanity-check noto: Barça (131) 2017 -> Y = +118.5M (vendita Neymar)
barca = node_season[(node_season.club_id == 131) & (node_season.season == 2017)]
print("\nBarça 2017:")
print(barca[["club_id", "season", "Y_margin_sum", "n_sales_Y", "spend_sum"]].to_string(index=False))

Tabella nodo-stagione (tutti i club): (4942, 9)
Club distinti: 1150

Barça 2017:
 club_id  season  Y_margin_sum  n_sales_Y   spend_sum
     131    2017   118500000.0        2.0 375100000.0


# Vista nodi CORE

In [13]:
# SEZIONE 5 — Vista nodi CORE
node_core = node_season.merge(core_keys, on=["club_id", "season"], how="inner").copy()

print("VISTA NODI core:", node_core.shape)
print("Club distinti:", node_core["club_id"].nunique())
print("Composizione Y:  Y>0:", int((node_core.Y_margin_sum > 0).sum()),
      "| Y<0:", int((node_core.Y_margin_sum < 0).sum()),
      "| Y=0:", int((node_core.Y_margin_sum == 0).sum()))

VISTA NODI core: (1649, 9)
Club distinti: 231
Composizione Y:  Y>0: 852 | Y<0: 508 | Y=0: 289


# Vista Archi

In [14]:
# SEZIONE 6 — Vista archi: edge list core-to-core
eco = owns[(owns["transfer_fee_amnt"].notna()) & (owns["transfer_fee_amnt"] > 0)].copy()
eco["source"] = eco["seller_id"]
eco["target"] = eco["buyer_id"]

# Tieni solo gli archi con ENTRAMBI gli estremi core in quella stagione
def both_core(s, t, yr):
    if pd.isna(s) or pd.isna(t) or pd.isna(yr):
        return False
    yr = int(yr)
    return (int(s), yr) in core_set and (int(t), yr) in core_set

mask = [both_core(s, t, yr) for s, t, yr in zip(eco["source"], eco["target"], eco["season"])]
edges_core = eco[pd.Series(mask, index=eco.index)].copy()

# Aggrega in edge list pesata, season-aware
edge_list = (edges_core
    .groupby(["season", "source", "target"])
    .agg(total_value=("transfer_fee_amnt", "sum"),
         n_transfers=("transfer_id", "count"))
    .reset_index())

surv = 100 * len(edges_core) / len(eco)
print(f"Archi economici totali:             {len(eco)}")
print(f"Archi core-to-core (singoli):       {len(edges_core)}")
print(f"Quota sopravvissuta al filtro core: {surv:.1f}%")
print(f"Archi aggregati (edge list):        {len(edge_list)}")
print(f"Stagioni coperte:                   {sorted(edge_list['season'].dropna().unique().tolist())}")

Archi economici totali:             9171
Archi core-to-core (singoli):       3586
Quota sopravvissuta al filtro core: 39.1%
Archi aggregati (edge list):        3374
Stagioni coperte:                   [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


*Sanity Check*

In [15]:
# Sanity-check: i top archi devono essere corridoi di mercato riconoscibili
print("Top 10 archi per valore:")
print(edge_list.sort_values("total_value", ascending=False).head(10).to_string(index=False))
# 131=Barça, 583=PSG: 131->583 2017 = Neymar 222M; 162->583 2018 = Mbappé 145M

Top 10 archi per valore:
 season  source  target  total_value  n_transfers
   2017     131     583  222000000.0            1
   2018     162     583  145000000.0            1
   2017      16     131  135000000.0            1
   2017      31     131  135000000.0            1
   2019     294      13  127200000.0            1
   2019      13     131  120000000.0            1
   2021     405     281  117500000.0            1
   2018     418     506  117000000.0            1
   2019     631     418  115000000.0            1
   2021      46     631  115000000.0            1


# Export csv

In [17]:
# SEZIONE 7 — Export per R/igraph
# Vista nodi
node_out = node_core.copy()
node_out["club_id"] = node_out["club_id"].astype(int)
node_out["season"]  = node_out["season"].astype(int)
node_out.to_csv(OUT_DIR / "node_season_core.csv", index=False)

# Vista archi
edge_out = edge_list.copy()
for c in ["source", "target", "season"]:
    edge_out[c] = edge_out[c].astype(int)
edge_out.to_csv(OUT_DIR / "edge_list_core.csv", index=False)

# Mappa id -> nome (da team e counter, per coprire tutti i nodi)
names_a = base[["team_id", "team_name"]].rename(columns={"team_id":"club_id","team_name":"club_name"})
names_b = base[["counter_team_id", "counter_team_name"]].rename(columns={"counter_team_id":"club_id","counter_team_name":"club_name"})
club_names = (pd.concat([names_a, names_b]).dropna().drop_duplicates("club_id").reset_index(drop=True))
club_names["club_id"] = pd.to_numeric(club_names["club_id"], errors="coerce").astype("Int64")
club_names = club_names.dropna(subset=["club_id"])
club_names.to_csv(OUT_DIR / "club_names.csv", index=False)

print("Esportati in", OUT_DIR.resolve())
print("  node_season_core.csv:", node_out.shape)
print("  edge_list_core.csv:  ", edge_out.shape)
print("  club_names.csv:      ", club_names.shape)

# Download da Colab
from google.colab import files
files.download(str(OUT_DIR / "node_season_core.csv"))
files.download(str(OUT_DIR / "edge_list_core.csv"))
files.download(str(OUT_DIR / "club_names.csv"))

Esportati in /content/outputs
  node_season_core.csv: (1649, 9)
  edge_list_core.csv:   (3374, 5)
  club_names.csv:       (3357, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>